In [1]:
import pandas as pd
import numpy as np

# Load raw data (ELT best practice)
df = pd.read_csv('../data/churn_prediction.csv')

df.head()


,customer_id,vintage,age,gender,dependents,occupation,city,customer_nw_category,branch_code,current_balance,...,average_monthly_balance_prevQ,average_monthly_balance_prevQ2,current_month_credit,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn,last_transaction
0,1,2101,66,Male,0.0,self_employed,187.0,2,755,1458.71,...,1458.71,1449.07,0.20,0.20,0.20,0.20,1458.71,1458.71,0,2019-05-21
1,2,2348,35,Male,0.0,self_employed,NaN,2,3214,5390.37,...,7799.26,12419.41,0.56,0.56,5486.27,100.56,6496.78,8787.61,0,2019-11-01
2,4,2194,31,Male,0.0,salaried,146.0,2,41,3913.16,...,4910.17,2815.94,0.61,0.61,6046.73,259.23,5006.28,5070.14,0,NaT
3,5,2329,90,NaN,NaN,self_employed,1020.0,2,582,2291.91,...,2084.54,1006.54,0.47,0.47,0.47,2143.33,2291.91,1669.79,1,2019-08-06
4,6,1579,42,Male,2.0,self_employed,1494.0,3,388,927.72,...,1643.31,1871.12,0.33,714.61,588.62,1538.06,1157.15,1677.16,1,2019-11-03


In [2]:
# Shape check
print("Initial Shape:", df.shape)

# Duplicate check
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

# Drop duplicates
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)


Initial Shape: (28382, 21)
Duplicate rows: 0
Shape after removing duplicates: (28382, 21)


In [3]:
# Drop irrelevant columns
df.drop(['customer_id', 'branch_code', 'city'], axis=1, inplace=True)

df.head()


,vintage,age,gender,dependents,occupation,customer_nw_category,current_balance,previous_month_end_balance,average_monthly_balance_prevQ,average_monthly_balance_prevQ2,current_month_credit,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn,last_transaction
0,2101,66,Male,0.0,self_employed,2,1458.71,1458.71,1458.71,1449.07,0.20,0.20,0.20,0.20,1458.71,1458.71,0,2019-05-21
1,2348,35,Male,0.0,self_employed,2,5390.37,8704.66,7799.26,12419.41,0.56,0.56,5486.27,100.56,6496.78,8787.61,0,2019-11-01
2,2194,31,Male,0.0,salaried,2,3913.16,5815.29,4910.17,2815.94,0.61,0.61,6046.73,259.23,5006.28,5070.14,0,NaT
3,2329,90,NaN,NaN,self_employed,2,2291.91,2291.91,2084.54,1006.54,0.47,0.47,0.47,2143.33,2291.91,1669.79,1,2019-08-06
4,1579,42,Male,2.0,self_employed,3,927.72,1401.72,1643.31,1871.12,0.33,714.61,588.62,1538.06,1157.15,1677.16,1,2019-11-03


In [4]:
# Gender → MODE (most frequent category)
df['gender'].fillna(df['gender'].mode()[0], inplace=True)

# Dependents → MEDIAN (robust to outliers)
df['dependents'].fillna(df['dependents'].median(), inplace=True)

# Occupation → label missing explicitly
df['occupation'].fillna('Unknown', inplace=True)

# Clean gender
df['gender'] = df['gender'].replace('Other', df['gender'].mode()[0])

df.isnull().sum()


vintage                           0
age                               0
gender                            0
dependents                        0
occupation                        0
customer_nw_category              0
current_balance                   0
previous_month_end_balance        0
average_monthly_balance_prevQ     0
average_monthly_balance_prevQ2    0
current_month_credit              0
previous_month_credit             0
current_month_debit               0
previous_month_debit              0
current_month_balance             0
previous_month_balance            0
churn                             0
last_transaction                  0
dtype: int64

In [5]:
# Monetary columns
money_cols = [
    'current_balance',
    'previous_month_end_balance',
    'average_monthly_balance_prevQ',
    'average_monthly_balance_prevQ2',
    'current_month_credit',
    'previous_month_credit',
    'current_month_debit',
    'previous_month_debit',
    'current_month_balance',
    'previous_month_balance'
]

# Replace negative values
for col in money_cols:
    df[col] = df[col].apply(lambda x: 0 if x < 0 else x)

# Cap outliers
for col in money_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

df[money_cols].describe()


,current_balance,previous_month_end_balance,average_monthly_balance_prevQ,average_monthly_balance_prevQ2,current_month_credit,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance
count,28382.000000,28382.000000,28382.000000,28382.000000,28382.000000,28382.000000,28382.000000,28382.000000,28382.000000,28382.000000
mean,6265.041971,6363.576965,6428.384442,6073.187975,1969.854510,2100.222339,2359.786381,2337.454280,6368.556560,6392.910128
std,9381.737713,9447.567053,8703.292408,8665.679021,6740.358583,7049.144139,7078.663619,6782.639757,9170.393828,9026.951153
min,47.689200,74.178200,1449.037700,121.648500,0.010000,0.010000,0.030000,0.030000,207.456300,320.083300
25%,1784.470000,1906.000000,2180.945000,1832.507500,0.310000,0.330000,0.410000,0.410000,1996.765000,2074.407500
50%,3281.255000,3379.915000,3542.865000,3359.600000,0.610000,0.630000,91.930000,109.960000,3447.995000,3465.235000
75%,6635.820000,6656.535000,6666.887500,6517.960000,707.272500,749.235000,1360.435000,1357.552500,6667.957500,6654.692500
max,64781.362600,66212.911600,60118.228800,59357.881000,50047.000400,51797.683900,51328.900400,48353.825500,64051.409200,62877.430200


In [6]:
# Convert to datetime
df['last_transaction'] = pd.to_datetime(df['last_transaction'], errors='coerce')

# Extract features
df['transaction_year'] = df['last_transaction'].dt.year
df['transaction_month'] = df['last_transaction'].dt.month
df['transaction_day'] = df['last_transaction'].dt.day

# Missing flag
df['no_transaction_flag'] = df['last_transaction'].isna().astype(int)

df[['last_transaction', 'transaction_year', 'transaction_month']].head()


,last_transaction,transaction_year,transaction_month
0,2019-05-21,2019.0,5.0
1,2019-11-01,2019.0,11.0
2,NaT,NaN,NaN
3,2019-08-06,2019.0,8.0
4,2019-11-03,2019.0,11.0


In [7]:
# Age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 40, 60, 100],
    labels=['Young', 'Adult', 'Middle Age', 'Senior'],
    include_lowest=True
)

# Balance segments
# Use np.inf as the final boundary so bins always increase after outlier capping.
df['balance_segment'] = pd.cut(
    df['current_balance'],
    bins=[-1, 1000, 10000, np.inf],
    labels=['Low', 'Medium', 'High']
)

# Total transactions
df['total_transactions'] = df['current_month_credit'] + df['current_month_debit']

# Activity level
# Use np.inf as the final boundary so the bucket logic remains stable for future data.
df['activity_level'] = pd.cut(
    df['total_transactions'],
    bins=[-1, 1000, 10000, 50000, np.inf],
    labels=['Low Activity', 'Moderate', 'High', 'Very High']
)

df[['age_group', 'balance_segment', 'activity_level']].head()


,age_group,balance_segment,activity_level
0,Senior,Medium,Low Activity
1,Adult,Medium,Moderate
2,Adult,Medium,Moderate
3,Senior,Medium,Low Activity
4,Middle Age,Low,Low Activity


In [8]:
# Final validation
print("Final Shape:", df.shape)

print("\nNegative values check:")
for col in money_cols:
    print(col, (df[col] < 0).sum())


Final Shape: (28382, 26)

Negative values check:
current_balance 0
previous_month_end_balance 0
average_monthly_balance_prevQ 0
average_monthly_balance_prevQ2 0
current_month_credit 0
previous_month_credit 0
current_month_debit 0
previous_month_debit 0
current_month_balance 0
previous_month_balance 0


In [9]:
df.to_csv('../data/churn_clean.csv', index=False)

print("Clean dataset saved successfully!")



Clean dataset saved successfully!
